# Exercise 17.1: Min–Max Scaling & Z-Score Standardisation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-17/01-minmax-and-zscore-scaling.ipynb)

**Reference:** Chapter 17 — Feature Scaling, Normalisation and Dimensionality Reduction (`chapters/feature-scaling-and-pca.tex`)  
**Estimated Time:** 15–20 min  

### Learning Objectives
- Implement min–max scaling $x_{\text{minmax}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$ and $z$-score standardisation $z = \frac{x - \mu}{\sigma}$.
- Reproduce the worked example on $\{10, 20, 30\}$ ($\mu = 20$, $\sigma \approx 8.165$, $z \approx (-1.225, 0, +1.225)$).
- Apply training-set scaling statistics to test data without data leakage.


In [ ]:
import numpy as np


### Mathematical Background

Raw datasets mix scales (e.g., age $18$–$80$ vs. income $20{,}000$–$200{,}000$). Two standard transformations bring features onto comparable ranges:

$$x_{\text{minmax}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}} \in [0, 1], \qquad z = \frac{x - \mu}{\sigma}$$

In Section 17.1.1 (Worked Example): for $\{10, 20, 30\}$, $\mu = 20$ and population $\sigma = \sqrt{200/3} \approx 8.165$, giving $z$-scores $\approx (-1.225, 0, +1.225)$.

**Crucial rule:** Always compute $\mu, \sigma, x_{\min}, x_{\max}$ on the **training set only**, then apply those exact statistics to validation/test data.


In [ ]:
def minmax_scale(
    X: np.ndarray,
    feature_min: np.ndarray | None = None,
    feature_max: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Scale features column-wise to [0, 1] using min-max normalisation.

    Formula:
        X_scaled = (X - feature_min) / (feature_max - feature_min)

    Args:
        X: 2D NumPy array of shape (n_samples, n_features).
        feature_min: Optional precomputed column minimums (shape (n_features,)).
        feature_max: Optional precomputed column maximums (shape (n_features,)).

    Returns:
        Tuple (X_scaled, feature_min, feature_max).
    """
    # TODO: Compute feature_min/feature_max along axis=0 if None, guard zero range, and return scaled array
    raise NotImplementedError("Complete this TODO")


def zscore_scale(
    X: np.ndarray,
    mean: np.ndarray | None = None,
    std: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Standardise features column-wise to zero mean and unit population variance.

    Formula:
        Z = (X - mean) / std  (using population std, ddof=0)

    Args:
        X: 2D NumPy array of shape (n_samples, n_features).
        mean: Optional precomputed column means.
        std: Optional precomputed column standard deviations.

    Returns:
        Tuple (X_scaled, mean, std).
    """
    # TODO: Compute mean/std along axis=0 if None, guard zero std, and return (X_scaled, mean, std)
    raise NotImplementedError("Complete this TODO")


<details>
<summary>Hint</summary>

- Compute statistics along `axis=0` so each feature column is scaled independently.
- Use `np.where(denom == 0.0, 1.0, denom)` to avoid division-by-zero on constant columns.
- Returning `(X_scaled, stat1, stat2)` allows you to pass the training statistics into subsequent calls for test data.

</details>


In [ ]:
# 1. Worked example Section 17.1.1: {10, 20, 30}
X_ex = np.array([[10.0], [20.0], [30.0]])
Z_ex, mu_ex, std_ex = zscore_scale(X_ex)
assert np.isclose(mu_ex[0], 20.0)
assert np.isclose(std_ex[0], 8.1649658, atol=1e-6)
assert np.allclose(Z_ex.ravel(), [-1.22474487, 0.0, 1.22474487], atol=1e-6)

# 2. Exercise 17.1 & 17.2
X_mm, _, _ = minmax_scale(np.array([[10.0], [15.0], [30.0]]))
assert np.isclose(X_mm[1, 0], 0.25)
Z_q2, _, _ = zscore_scale(np.array([[25.0]]), mean=np.array([20.0]), std=np.array([2.5]))
assert np.isclose(Z_q2[0, 0], 2.0)

# 3. Multi-feature train/test scaling without leakage
X_train = np.array([[20.0, 30000.0], [40.0, 60000.0], [60.0, 90000.0], [80.0, 120000.0]])
X_test = np.array([[50.0, 150000.0]])
Z_tr, tr_mu, tr_std = zscore_scale(X_train)
Z_te, _, _ = zscore_scale(X_test, mean=tr_mu, std=tr_std)
assert np.allclose(Z_tr.mean(axis=0), 0.0, atol=1e-9)
assert np.allclose(Z_tr.std(axis=0), 1.0, atol=1e-9)
assert np.isclose(Z_te[0, 0], 0.0)
assert Z_te[0, 1] > 1.0  # 150k is above train max 120k

print('All Chapter 17.1 assertions passed!')


### Think Deeper

1. Why does fitting `zscore_scale` on the combined train+test dataset cause data leakage?
2. When might you prefer `zscore_scale` over `minmax_scale` if your dataset contains extreme outliers?
